# Day 6 — Feature Engineering & Preprocessing Pipeline

## Objectives
- Separate features (X) and target (y)
- Identify numeric and categorical columns
- Build a scikit-learn ColumnTransformer + Pipeline
- Fit preprocessor only on training data (no leakage)
- Transform train and test sets
- Create simple engineered features
- Save the fitted preprocessor for reuse

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import joblib

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

print("Libraries imported successfully")

Libraries imported successfully


In [2]:
current_directory = Path.cwd()

if (current_directory / "data").exists():
    PROJECT_ROOT = current_directory
elif (current_directory.parent / "data").exists():
    PROJECT_ROOT = current_directory.parent
else:
    raise FileNotFoundError(
        "Could not find project root. Open the correct folder in VS Code."
    )

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
MODELS_DIR = PROJECT_ROOT / "models"
REPORTS_DIR = PROJECT_ROOT / "reports"
FIGURES_DIR = REPORTS_DIR / "figures"

MODELS_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_PATH = PROCESSED_DIR / "train_churn_data.csv"
TEST_PATH = PROCESSED_DIR / "test_churn_data.csv"

print("Project root:", PROJECT_ROOT)
print("Train path exists:", TRAIN_PATH.exists())
print("Test path exists:", TEST_PATH.exists())

Project root: c:\Users\gurmeet singh\Projects\customer-churn-retention-engine
Train path exists: True
Test path exists: True


In [3]:
train_df = pd.read_csv(TRAIN_PATH)
test_df = pd.read_csv(TEST_PATH)

print("Train shape:", train_df.shape)
print("Test shape :", test_df.shape)
print("\nTrain columns:")
print(list(train_df.columns))

Train shape: (5616, 20)
Test shape : (1405, 20)

Train columns:
['gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'MonthlyCharges', 'TotalCharges', 'Churn']


In [4]:
TARGET = "Churn"

X_train = train_df.drop(columns=[TARGET])
y_train = train_df[TARGET]

X_test = test_df.drop(columns=[TARGET])
y_test = test_df[TARGET]

print("X_train shape:", X_train.shape)
print("y_train shape:", y_train.shape)
print("X_test shape :", X_test.shape)
print("y_test shape :", y_test.shape)

print("\nTarget distribution (train):")
print(y_train.value_counts(normalize=True).round(3))

X_train shape: (5616, 19)
y_train shape: (5616,)
X_test shape : (1405, 19)
y_test shape : (1405,)

Target distribution (train):
Churn
0    0.736
1    0.264
Name: proportion, dtype: float64


In [5]:
numeric_features = X_train.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

categorical_features = X_train.select_dtypes(
    include=["object"]
).columns.tolist()

print("Numeric features (", len(numeric_features), "):")
print(numeric_features)

print("\nCategorical features (", len(categorical_features), "):")
print(categorical_features)

Numeric features ( 4 ):
['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges']

Categorical features ( 15 ):
['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod']


In [6]:
def add_engineered_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    Create simple, interpretable features.
    The same transformations must be applied to train and test data.
    """
    df = df.copy()

    # Average monthly charge
    df["AvgMonthlyCharge"] = np.where(
        df["tenure"] > 0,
        df["TotalCharges"] / df["tenure"],
        df["MonthlyCharges"]
    )

    # Tenure groups
    df["TenureGroup"] = pd.cut(
        df["tenure"],
        bins=[-1, 12, 24, 48, 72],
        labels=[
            "0-12 months",
            "13-24 months",
            "25-48 months",
            "49-72 months"
        ]
    )

    return df

In [7]:
X_train = add_engineered_features(X_train)
X_test = add_engineered_features(X_test)

new_columns = [
    col for col in X_train.columns
    if col not in train_df.columns
]

print("New columns added:")
print(new_columns)

print("\nX_train shape after engineering:", X_train.shape)
print("X_test shape after engineering :", X_test.shape)

New columns added:
['AvgMonthlyCharge', 'TenureGroup']

X_train shape after engineering: (5616, 21)
X_test shape after engineering : (1405, 21)


In [8]:
numeric_features = X_train.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

categorical_features = X_train.select_dtypes(
    include=["object", "category"]
).columns.tolist()

print("Updated Numeric features:")
print(numeric_features)

print("\nUpdated Categorical features:")
print(categorical_features)

Updated Numeric features:
['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges', 'AvgMonthlyCharge']

Updated Categorical features:
['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'TenureGroup']


In [9]:
from sklearn.model_selection import train_test_split

In [10]:
# Numeric pipeline
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

# Categorical pipeline
categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(
        handle_unknown="ignore",
        sparse_output=False
    ))
])

# Combine both pipelines
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ],
    remainder="drop"
)

print("Preprocessor created successfully")

Preprocessor created successfully


In [11]:
preprocessor

,transformers,"[('num', ...), ('cat', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True
,force_int_remainder_cols,'deprecated'
,missing_values,nan
,strategy,'median'
,fill_value,None


In [12]:
# Fit ONLY on training data
preprocessor.fit(X_train)

print("Preprocessor fitted successfully on training data")

Preprocessor fitted successfully on training data


In [13]:
X_train_processed = preprocessor.transform(X_train)
X_test_processed = preprocessor.transform(X_test)

print("X_train_processed shape:", X_train_processed.shape)
print("X_test_processed shape :", X_test_processed.shape)
print("Type:", type(X_train_processed))

X_train_processed shape: (5616, 50)
X_test_processed shape : (1405, 50)
Type: <class 'numpy.ndarray'>


In [14]:
feature_names = preprocessor.get_feature_names_out()

print("Number of final features:", len(feature_names))

print("\nFirst 15 feature names:")
print(feature_names[:15])

print("\nLast 10 feature names:")
print(feature_names[-10:])

Number of final features: 50

First 15 feature names:
['num__SeniorCitizen' 'num__tenure' 'num__MonthlyCharges'
 'num__TotalCharges' 'num__AvgMonthlyCharge' 'cat__gender_Female'
 'cat__gender_Male' 'cat__Partner_No' 'cat__Partner_Yes'
 'cat__Dependents_No' 'cat__Dependents_Yes' 'cat__PhoneService_No'
 'cat__PhoneService_Yes' 'cat__MultipleLines_No'
 'cat__MultipleLines_No phone service']

Last 10 feature names:
['cat__PaperlessBilling_No' 'cat__PaperlessBilling_Yes'
 'cat__PaymentMethod_Bank transfer (automatic)'
 'cat__PaymentMethod_Credit card (automatic)'
 'cat__PaymentMethod_Electronic check' 'cat__PaymentMethod_Mailed check'
 'cat__TenureGroup_0-12 months' 'cat__TenureGroup_13-24 months'
 'cat__TenureGroup_25-48 months' 'cat__TenureGroup_49-72 months']


In [15]:
X_train_df = pd.DataFrame(
    X_train_processed,
    columns=feature_names
)

X_test_df = pd.DataFrame(
    X_test_processed,
    columns=feature_names
)

print("Processed training data:")
display(X_train_df.head(3))

Processed training data:


,num__SeniorCitizen,num__tenure,num__MonthlyCharges,num__TotalCharges,num__AvgMonthlyCharge,cat__gender_Female,cat__gender_Male,cat__Partner_No,cat__Partner_Yes,cat__Dependents_No,...,cat__PaperlessBilling_No,cat__PaperlessBilling_Yes,cat__PaymentMethod_Bank transfer (automatic),cat__PaymentMethod_Credit card (automatic),cat__PaymentMethod_Electronic check,cat__PaymentMethod_Mailed check,cat__TenureGroup_0-12 months,cat__TenureGroup_13-24 months,cat__TenureGroup_25-48 months,cat__TenureGroup_49-72 months
0,-0.440315,-1.241331,0.193165,-0.944714,0.179602,0.0,1.0,1.0,0.0,1.0,...,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0
1,-0.440315,-0.711078,0.647355,-0.434062,0.726230,1.0,0.0,1.0,0.0,1.0,...,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0
2,-0.440315,1.409933,0.820380,1.794294,1.000568,0.0,1.0,0.0,1.0,0.0,...,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0


In [16]:
print("Missing values in processed train data:",
      X_train_df.isna().sum().sum())

print("Missing values in processed test data:",
      X_test_df.isna().sum().sum())

Missing values in processed train data: 0
Missing values in processed test data: 0


In [17]:
preprocessor_path = MODELS_DIR / "preprocessor.joblib"

joblib.dump(
    preprocessor,
    preprocessor_path
)

print("Preprocessor saved to:", preprocessor_path)

Preprocessor saved to: c:\Users\gurmeet singh\Projects\customer-churn-retention-engine\models\preprocessor.joblib


In [18]:
np.save(
    PROCESSED_DIR / "X_train_processed.npy",
    X_train_processed
)

np.save(
    PROCESSED_DIR / "X_test_processed.npy",
    X_test_processed
)

np.save(
    PROCESSED_DIR / "y_train.npy",
    y_train.to_numpy()
)

np.save(
    PROCESSED_DIR / "y_test.npy",
    y_test.to_numpy()
)

print("Processed arrays saved successfully.")

Processed arrays saved successfully.


In [19]:
feature_names_path = REPORTS_DIR / "final_feature_names.csv"

pd.Series(feature_names).to_csv(
    feature_names_path,
    index=False,
    header=["feature_name"]
)

print("Feature names saved to:", feature_names_path)

Feature names saved to: c:\Users\gurmeet singh\Projects\customer-churn-retention-engine\reports\final_feature_names.csv


In [20]:
print("========== DAY 6 PREPROCESSING CHECK ==========")

print("Train rows:", X_train_processed.shape[0])
print("Test rows :", X_test_processed.shape[0])
print("Final features:", X_train_processed.shape[1])

print("Train NaNs:", np.isnan(X_train_processed).sum())
print("Test NaNs :", np.isnan(X_test_processed).sum())

print("Preprocessor exists:", preprocessor_path.exists())
print("Feature names file exists:", feature_names_path.exists())

========== DAY 6 PREPROCESSING CHECK ==========
Train rows: 5616
Test rows : 1405
Final features: 50
Train NaNs: 0
Test NaNs : 0
Preprocessor exists: True
Feature names file exists: True
